# Лабораторная работа №5. Дерево решений

**Цель работы:** Реализовать алгоритм построения бинарного дерева решений для задачи классификации:
1. Реализовать функцию расчета критерия Джини (`gini`).
2. Реализовать расчет критерия прироста информации (`gain`).
3. Реализовать критерии останова (не менее двух: ограничение максимальной глубины `max_depth`, минимальное количество объектов в листе `min_samples_leaf`, отсутствие прироста качества).
4. Реализовать функцию подсчета метрики точности модели (`accuracy_metric`).
5. Проверить работу самописного дерева для решения задачи классификации на датасете, сгенерированном с помощью `make_classification`, сравнив результаты с `DecisionTreeClassifier` из `sklearn` (результаты должны совпасть полностью).

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

---
## Классы узла (`Node`) и терминального узла (`Leaf`)

In [16]:
# Реализуем класс узла
class Node:
    def __init__(self, index, t, true_branch, false_branch):
        self.index = index  # индекс признака, по которому ведется сравнение с порогом в этом узле
        self.t = t  # значение порога
        self.true_branch = true_branch  # поддерево, удовлетворяющее условию в узле (<= t)
        self.false_branch = false_branch  # поддерево, не удовлетворяющее условию в узле (> t)


# И класс терминального узла (листа)
class Leaf:
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
        self.prediction = self.predict()

    def predict(self):
        # подсчет количества объектов разных классов
        classes = {}  # сформируем словарь "класс: количество объектов"
        for label in self.labels:
            if label not in classes:
                classes[label] = 0
            classes[label] += 1

        # найдем класс, количество объектов которого будет максимальным в этом листе
        # при равенстве голосов выбираем наименьший класс (для гарантированного совпадения со sklearn)
        max_count = max(classes.values())
        best_classes = [cls for cls, count in classes.items() if count == max_count]
        prediction = min(best_classes)
        return prediction

---
## 1. Расчет критерия Джини

In [17]:
# 1) Расчет критерия Джини
def gini(labels):
    """
    G = 1 - sum(p_i^2), где p_i — доля объектов класса i
    """
    if len(labels) == 0:
        return 0.0

    classes = {}
    for label in labels:
        if label not in classes:
            classes[label] = 0
        classes[label] += 1

    n_samples = len(labels)
    gini_value = 1.0
    for label in classes:
        p = classes[label] / n_samples
        gini_value -= p ** 2

    return max(0.0, float(gini_value))

---
## 2. Расчет прироста информации и разбиение датасета

In [18]:
# 2) Расчет прироста информации
def gain(left_labels, right_labels, root_gini):
    """
    Information Gain = Gini(parent) - (n_left/n * Gini(left) + n_right/n * Gini(right))
    """
    n = len(left_labels) + len(right_labels)
    if n == 0:
        return 0.0

    p_left = len(left_labels) / n
    p_right = len(right_labels) / n

    child_gini = p_left * gini(left_labels) + p_right * gini(right_labels)
    return float(root_gini - child_gini)


# Разбиение датасета в узле
def split(data, labels, column_index, t):
    left = np.where(data[:, column_index] <= t)
    right = np.where(data[:, column_index] > t)

    true_data = data[left]
    false_data = data[right]

    true_labels = labels[left]
    false_labels = labels[right]

    return true_data, false_data, true_labels, false_labels

---
## 3. Критерии останова и построение дерева

Реализованы следующие критерии останова:
1. **`max_depth`**: ограничение максимальной глубины дерева.
2. **`min_samples_leaf`**: минимальное количество объектов в листе (проверяется в `find_best_split`).
3. **`gain == 0`**: прекращение рекурсии при отсутствии прироста в качестве (чистота узла или невозможность улучшения).

In [19]:
# 3) Нахождение наилучшего разбиения
def find_best_split(data, labels, min_samples_leaf=3):
    # Обозначим минимальное количество объектов в листе (критерий останова)
    root_gini = gini(labels)

    best_gain = 0.0
    best_t = None
    best_index = None

    n_features = data.shape[1]

    for index in range(n_features):
        t_values = np.unique(data[:, index])
        if len(t_values) < 2:
            continue

        # Пороги перебираются как середины между соседними уникальными значениями (как в sklearn)
        thresholds = (t_values[:-1] + t_values[1:]) / 2.0

        for t in thresholds:
            true_data, false_data, true_labels, false_labels = split(data, labels, index, t)

            # Критерий останова: отсекаем разбиения, оставляющие меньше min_samples_leaf объектов
            if len(true_labels) < min_samples_leaf or len(false_labels) < min_samples_leaf:
                continue

            current_gain = gain(true_labels, false_labels, root_gini)

            if current_gain > best_gain:
                best_gain, best_t, best_index = current_gain, t, index

    return best_gain, best_t, best_index


# Построение дерева с помощью рекурсивной функции
def build_tree(data, labels, max_depth=None, min_samples_leaf=3, depth=0):
    # Критерий останова 1: ограничение максимальной глубины
    if max_depth is not None and depth >= max_depth:
        return Leaf(data, labels)

    # Критерий останова: узел уже чист
    if len(np.unique(labels)) <= 1:
        return Leaf(data, labels)

    gain_val, t, index = find_best_split(data, labels, min_samples_leaf=min_samples_leaf)

    # Критерий останова 2: базовый случай — прекращаем рекурсию, когда нет прироста в качестве
    if gain_val == 0 or t is None:
        return Leaf(data, labels)

    true_data, false_data, true_labels, false_labels = split(data, labels, index, t)

    # Рекурсивно строим два поддерева
    true_branch = build_tree(true_data, true_labels, max_depth=max_depth, min_samples_leaf=min_samples_leaf, depth=depth + 1)
    false_branch = build_tree(false_data, false_labels, max_depth=max_depth, min_samples_leaf=min_samples_leaf, depth=depth + 1)

    # Возвращаем класс узла со всеми поддеревьями, то есть целого дерева
    return Node(index, t, true_branch, false_branch)

---
## Классификация объектов, предсказание и вывод структуры дерева

In [20]:
def classify_object(obj, node):
    # Останавливаем рекурсию, если достигли листа
    if isinstance(node, Leaf):
        answer = node.prediction
        return answer

    if obj[node.index] <= node.t:
        return classify_object(obj, node.true_branch)
    else:
        return classify_object(obj, node.false_branch)


def predict(data, tree):
    classes = []
    for obj in data:
        prediction = classify_object(obj, tree)
        classes.append(prediction)
    return classes


# Напечатаем ход нашего дерева
def print_tree(node, spacing=""):
    # Если лист, то выводим его прогноз
    if isinstance(node, Leaf):
        print(spacing + "Прогноз:", node.prediction)
        return

    # Выведем значение индекса и порога на этом узле
    print(spacing + "Индекс", str(node.index), "<=", str(node.t))

    # Рекурсионный вызов функции на положительном поддереве
    print(spacing + "--> True:")
    print_tree(node.true_branch, spacing + "  ")

    # Рекурсионный вызов функции на отрицательном поддереве
    print(spacing + "--> False:")
    print_tree(node.false_branch, spacing + "  ")

---
## 4. Функция подсчета метрики точности модели

In [21]:
# 4) Введем функцию подсчета точности как доли правильных ответов
def accuracy_metric(actual, predicted):
    actual = np.array(actual)
    predicted = np.array(predicted)
    if len(actual) == 0:
        return 0.0
    return float(np.mean(actual == predicted))

---
## 5. Проверка работы самописного дерева и сравнение с `DecisionTreeClassifier` из sklearn

In [22]:
# 5) Генерируем датасет с помощью make_classification
X, y = make_classification(
    n_samples=300,
    n_features=5,
    n_informative=4,
    n_redundant=1,
    n_classes=2,
    flip_y=0.0,
    class_sep=1.5,
    random_state=42
)
X = np.ascontiguousarray(X, dtype=np.float32)

# Разделение на train и test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Критерии останова
max_depth = 3
min_samples_leaf = 3

# 1. Обучение самописного дерева
my_tree = build_tree(X_train, y_train, max_depth=max_depth, min_samples_leaf=min_samples_leaf)
my_pred = predict(X_test, my_tree)

# 2. Обучение дерева решений из sklearn с теми же критериями останова
sk_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=max_depth,
    min_samples_leaf=min_samples_leaf,
    random_state=42
)
sk_tree.fit(X_train, y_train)
sk_pred = sk_tree.predict(X_test)

# 3. Сравнение результатов
my_acc = accuracy_metric(y_test, my_pred)
sk_acc = accuracy_metric(y_test, sk_pred)
matches = np.sum(np.array(my_pred) == np.array(sk_pred))

print(f"Accuracy самописного дерева: {my_acc:.5f}")
print(f"Accuracy sklearn:            {sk_acc:.5f}")
print(f"Количество совпадающих предсказаний: {matches} из {len(y_test)} ({matches / len(y_test) * 100:.1f}%)")

assert matches == len(y_test), "Предсказания не совпали!"
assert my_acc == sk_acc, "Метрики Accuracy не равны!"
print("✅ Результаты самописного дерева и sklearn совпали ПОЛНОСТЬЮ (100%)!")

print("\nСтруктура построенного дерева решений:")
print_tree(my_tree)

Accuracy самописного дерева: 0.96667
Accuracy sklearn:            0.96667
Количество совпадающих предсказаний: 90 из 90 (100.0%)
✅ Результаты самописного дерева и sklearn совпали ПОЛНОСТЬЮ (100%)!

Структура построенного дерева решений:
Индекс 3 <= -0.24450232
--> True:
  Индекс 1 <= 0.47461012
  --> True:
    Прогноз: 0
  --> False:
    Индекс 2 <= -1.7008083
    --> True:
      Прогноз: 0
    --> False:
      Прогноз: 1
--> False:
  Индекс 2 <= -1.8710352
  --> True:
    Прогноз: 0
  --> False:
    Прогноз: 1


---
## Вывод

В ходе лабораторной работы №5 были успешно реализованы все пункты задания:
1. Функция расчета критерия неопределенности Джини (`gini`).
2. Функция расчета прироста информации (`gain`) при разбиении узла.
3. Критерии останова ветвления дерева: ограничение максимальной глубины (`max_depth`), минимальное количество объектов в листе (`min_samples_leaf`), а также остановка при отсутствии прироста качества (`gain == 0`).
4. Функция оценки точности классификации (`accuracy_metric`).
5. Проведено сравнение с реализацией `DecisionTreeClassifier` из библиотеки `scikit-learn`: результаты предсказаний и метрика качества совпали на 100%.